In [ ]:
import math
import os
import sys
from pathlib import Path

import einops
import numpy as np
import torch as t
from torch import Tensor
import plotly.graph_objects as go
from IPython.display import display
from ipywidgets import interact
from jaxtyping import Bool, Float

# Make sure exercises are in the path.
# Walk up from the notebook's cwd until we find the ARENA/ folder that
# contains ARENA_3.0 — this way the notebook keeps working no matter how
# deep it's nested under scratchpad_notebooks/ (e.g. scratchpad_notebooks/1/).
def find_arena_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ARENA_3.0").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find an ARENA_3.0 folder above {start}")

root_dir = find_arena_root(Path.cwd())
exercises_dir = root_dir / "ARENA_3.0" / "chapter0_fundamentals" / "exercises"
section_dir = exercises_dir / "part1_ray_tracing"

print(exercises_dir)
print(exercises_dir.exists())         # should be True now
print(section_dir.exists())           # should be True now

if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

# ARENA's own solutions.py files locate the course root via
# `next(p for p in Path.cwd().parents if (p / chapter).exists())`,
# which only works if the process's cwd is physically inside
# .../exercises/<section>/ (the normal ARENA layout). Since we're running
# from scratchpad_notebooks/ instead, chdir into section_dir so that
# lookup succeeds instead of raising StopIteration on import.
os.chdir(section_dir)

import part1_ray_tracing.tests as tests
from part1_ray_tracing.utils import (
    render_lines_with_plotly,
    setup_widget_fig_ray,
    setup_widget_fig_triangle,
)
from plotly_utils import imshow

MAIN = __name__ == "__main__"

In [7]:
def make_rays_1d(num_pixels: int, y_limit: float) -> Tensor:
    """
    num_pixels: The number of pixels in the y dimension. Since there is one ray per pixel, this is
        also the number of rays.
    y_limit: At x=1, the rays should extend from -y_limit to +y_limit, inclusive of both endpoints.

    Returns: shape (num_pixels, num_points=2, num_dim=3) where the num_points dimension contains
        (origin, direction) and the num_dim dimension contains xyz.

    Example of make_rays_1d(9, 1.0): [
        [[0, 0, 0], [1, -1.0, 0]],
        [[0, 0, 0], [1, -0.75, 0]],
        [[0, 0, 0], [1, -0.5, 0]],
        ...
        [[0, 0, 0], [1, 0.75, 0]],
        [[0, 0, 0], [1, 1, 0]],
    ]
    """
    
    rays = t.zeros((num_pixels, 2, 3))
    t.linspace(-y_limit, y_limit, num_pixels, out=rays[:, 1, 1])
    rays[:, 1, 0] = 1
    return rays


rays1d = make_rays_1d(9, 10.0)
fig = render_lines_with_plotly(rays1d)

In [16]:
fig: go.FigureWidget = setup_widget_fig_ray()
display(fig)


@interact(v=(0.0, 6.0, 0.01), seed=(0, 10, 1))
def update(v=0.0, seed=0):
    t.manual_seed(seed)
    L_1, L_2 = t.rand(2, 2)
    P = lambda v: L_1 + v * (L_2 - L_1)
    x, y = zip(P(0), P(6))
    with fig.batch_update():
        fig.update_traces({"x": x, "y": y}, 0)
        fig.update_traces({"x": [L_1[0], L_2[0]], "y": [L_1[1], L_2[1]]}, 1)
        fig.update_traces({"x": [P(v)[0]], "y": [P(v)[1]]}, 2)

FigureWidget({
    'data': [{'type': 'scatter', 'uid': '3801bcfd-7878-4a28-bc2f-617baf3d5147', 'x': [], 'y': []},
             {'marker': {'size': 12},
              'mode': 'markers',
              'name': 'v=0',
              'type': 'scatter',
              'uid': '1fc0402c-ca65-4421-a572-80456c701d79',
              'x': [],
              'y': []},
             {'marker': {'size': 12, 'symbol': 'x'},
              'mode': 'markers',
              'name': 'v=1',
              'type': 'scatter',
              'uid': '34d925eb-f58c-4e73-b7b7-20516e3b6abc',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 'l': 40, 't': 60},
               'showlegend': False,
               'template': '...',
               'title': {'text': 'Ray coordinates illustration'},
               'width': 500,
               'xaxis': {'range': [-1.5, 2.5]},
               'yaxis': {'range': [-1.5, 2.5]}}
})

interactive(children=(FloatSlider(value=0.0, description='v', max=6.0, step=0.01), IntSlider(value=0, descript…

In [23]:
def intersect_ray_1d(ray: Float[Tensor, "points dims"], segment: Float[Tensor, "points dims"]) -> bool:
    """
    ray: shape (n_points=2, n_dim=3)  # O, D points
    segment: shape (n_points=2, n_dim=3)  # L_1, L_2 points

    Return True if the ray intersects the segment.
    """
    
    ray = ray[:, :2]
    segment = segment[:, :2]

    # Ray is [[Ox, Oy], [Dx, Dy]]
    O, D = ray
    # Segment is [[L1x, L1y], [L2x, L2y]]
    L_1, L_2 = segment

    # Create matrix and vector, and solve equation
    mat = t.stack([D, L_1 - L_2], dim=-1)
    vec = L_1 - O

    # Solve equation (return False if no solution)
    try:
        sol = t.linalg.solve(mat, vec)
    except RuntimeError:
        return False

    # If there is a solution, check the soln is in the correct range for there to be an intersection
    u = sol[0].item()
    v = sol[1].item()
    return (u >= 0.0) and (v >= 0.0) and (v <= 1.0)
    

tests.test_intersect_ray_1d(intersect_ray_1d)
tests.test_intersect_ray_1d_special_case(intersect_ray_1d)

StopIteration: 